# S3 Explorer

A file explorer for an S3 bucket, running in this notebook. Browse folders, preview files, upload
(drag and drop works), download, rename and delete — with a short-lived credential from **STS
`AssumeRoleWithWebIdentity`**, issued after you sign in with your **OpenID Connect** provider.

```
 you ──sign in──▶ OIDC provider ──ID token──▶ STS ──▶ temporary credential (1 hour, one role)
                                                          │
                                                          ▼
                                               any standard S3 client
                                                          │   · this notebook's explorer (boto3 underneath)
                                                          │   · boto3, pandas or Spark in your own code
                                                          │   · the AWS CLI, s3cmd, rclone
                                                          │   · desktop apps such as S3 Browser or Cyberduck
                                                          │
                                                          │   S3 requests signed with the credential
                                                          ▼
                                                     S3 endpoint ──▶ object storage
```

The explorer is just one S3 client. Any tool that lets you set the endpoint URL and a session token
can use the same credential: point it at the S3 endpoint with path-style addressing. What the
explorer can do is decided by the role, which the S3 endpoint checks on every request.

**The credential is cached.** The STS session token is a JWT; its `exp` claim is when the S3
endpoint stops accepting it. The notebook keeps the credential in `~/.cache/s3-assumerole/`
(readable only by you) until then, so re-running the notebook or restarting the kernel within the
hour skips the sign-in. When it does expire, the explorer locks and its **Renew** button signs you
in again in place.

### Before you start

- `pip install -r requirements.txt` in the notebook's kernel, and the `s3_explorer/` folder next to
  this notebook.
- An OAuth client at your OpenID Connect provider that may use the device authorization grant
  (RFC 8628). Ideally a public client: it has no secret, so it cannot use the
  client-credentials grant.
- An STS that accepts that provider's tokens in `AssumeRoleWithWebIdentity`, and a role for you.
- Direct network access to the provider, STS and the S3 endpoint. The notebook ignores
  `http_proxy`/`https_proxy`/`no_proxy` and never uses a proxy.
- A config file in `./.secrets/` or `../.secrets/`. It holds the client and every setting, so
  nothing environment-specific is in the notebook. The secret is never printed.

The config file, e.g. `.secrets/credentials.json`:

```json
{
  "client_id":     "<OAuth client id>",
  "client_secret": "<OAuth client secret>",
  "oidc_issuer":   "https://login.example.com",
  "sts_endpoint":  "https://sts.example.com/",
  "s3_endpoint":   "https://s3.example.com/",
  "role_arn":      "arn:aws:iam::123456789012:role/s3-explorer",
  "bucket":        "my-bucket"
}
```

- `client_secret`: optional. A public client (RFC 8628) has none; include it if your provider
  requires a secret for the device grant.
- `oidc_issuer`: the provider's issuer URL. Its endpoints come from
  `<oidc_issuer>/.well-known/openid-configuration`.
- `role_arn`: the full ARN of the role to assume, sent to STS as is. For read-only access, use a
  config whose `role_arn` is a read-only role.
- `bucket`: the bucket the explorer opens.

Optional: `provider_id` (the `ProviderId` parameter of `AssumeRoleWithWebIdentity`, if your STS
wants one) and `extra_buckets` (more buckets to offer in the explorer's bucket box).

Instead of the file, you can put the same JSON, base64-encoded, in the environment variable
`S3_ASSUMEROLE_CONFIG_B64` of the shell that starts Jupyter. When it is set it wins over the file:

```bash
export S3_ASSUMEROLE_CONFIG_B64=$(base64 -w0 .secrets/credentials.json)
jupyter lab
```

The tier shown in the explorer comes from the URLs: **DEV**, **ALPHA** or **BETA** if `sts_endpoint`
or `s3_endpoint` contains `.dev.`, `.alpha.` or `.beta.`, otherwise **PROD**.

Run the cells top to bottom (**Run → Run All Cells**). Sign-in pauses until you approve it in the
browser, unless a cached credential is still valid.

## Settings

`CONFIG` is the config file, looked up in `./.secrets/` then `../.secrets/`;
`S3_ASSUMEROLE_CONFIG_B64` replaces it when set. What the explorer may do is decided by the role in
the config: with a read-only role, uploads, renames and deletes are refused by the S3 endpoint, and
the explorer shows the refusal.

In [ ]:
import s3_explorer

CONFIG = "credentials.json"        # 👈 the config file in .secrets/ ($S3_ASSUMEROLE_CONFIG_B64 wins if set)

session = s3_explorer.AssumeRoleSession(s3_explorer.find_config(CONFIG))
cfg = session.cfg

print(f"Environment: {cfg['env'].upper()}  (from {session.config_source})")
print(f"  Sign-in   {cfg['oidc_issuer']}")
print(f"  STS       {cfg['sts_endpoint']}")
print(f"  S3        {cfg['s3_endpoint']}")
print(f"  Role      {session.role_arn}")
print(f"  Bucket    s3://{cfg['bucket']}")

## Sign in

Uses the cached credential if it is still valid. Otherwise a browser tab opens on your provider's
sign-in page (the link is shown below too); approve it and the cell finishes by itself.

In [ ]:
s3_explorer.show_sign_in(session)

## Explorer

| To | Do |
| --- | --- |
| Open a folder | Double-click it, or select it and press **Enter** |
| Preview a file | Click it (CSV as a table, JSON, text, images, PDF). **Details** shows its metadata |
| Go somewhere else | Breadcrumbs, **← → ↑**, or click the path box and type `s3://bucket/prefix` |
| Upload | **Upload** / **Folder**, or drop files and folders anywhere on the explorer |
| Download | Select and **Download**. Several files or a folder come as one `.zip` |
| Select several | **Ctrl**/**Shift**-click, the checkboxes, or **Ctrl+A** |
| Rename or move | **Rename** (or **F2**). Type a path to move it within the bucket |
| Delete | **Delete** (or the **Delete** key). Folders go with everything in them |
| Get a new credential | **Renew** in the header. It signs you in again without leaving the explorer |
| Forget the cached credential | **Sign out** in the header (or `session.sign_out()` in a cell) |

Every action is a plain S3 call to the S3 endpoint, checked against the role. Type a bucket the
role doesn't cover in the bucket box to see the S3 endpoint refuse it.

In [ ]:
explorer = s3_explorer.S3Explorer(session, cfg["bucket"], buckets=cfg["extra_buckets"])
explorer